# S&P500 50-Stock Panel EDA

This output-stripped notebook is a guarded exploratory analysis for the experimental local-only S&P500 50-stock panel. It does not download data, train models, evaluate checkpoints, or update the public registry by default.

The benchmark note is `docs/benchmarks/sp500_50_panel.md`. The notebook can inspect a local processed panel when present, and it must also pass when no local data exists.

## Data-use caveat

The 50-stock panel uses optional `yfinance` access to Yahoo-backed adjusted-close data. `yfinance` is unaffiliated with Yahoo. Downloaded data is for local research and educational use subject to Yahoo's terms, and raw or processed market data must not be redistributed or committed.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

RUN_DOWNLOAD = False
RUN_TRAINING = False
RUN_EVALUATION = False
ALLOW_MISSING_OUTPUTS = True

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.parent != REPO_ROOT and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-time-causal-vae")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from scripts.download_sp500_50_panel import SECTOR_ETFS, SECTOR_TICKERS, UNIVERSE_ID

from time_causal_vae.experiments.multidim_profiles import select_multidim_profile

PROCESSED_DIR = Path(
    os.environ.get(
        "SP50050_PANEL_PROCESSED_DIR",
        str(REPO_ROOT / "data" / "processed" / "sp500_50_panel"),
    )
)
METADATA_PATH = PROCESSED_DIR / "metadata.json"
RAW_DATA_PATH = PROCESSED_DIR / "raw_data.pt"
LABELS_PATH = PROCESSED_DIR / "labels.pt"

## Static ticker and sector universe

The first public universe is static and sector-stratified. The downloader should update the versioned universe explicitly if a ticker materially shortens the aligned panel; it should not silently substitute another asset.

In [2]:
universe_rows = []
for sector_name, tickers in SECTOR_TICKERS.items():
    for ticker in tickers:
        universe_rows.append({"sector": sector_name, "ticker": ticker})

universe = pd.DataFrame(universe_rows)
display(Markdown(f"Universe id: `{UNIVERSE_ID}`; stock count: `{len(universe)}`."))
display(universe.groupby("sector").size().rename("ticker_count").reset_index())
display(universe.head(12))
display(pd.DataFrame([{"sector": sector, "etf": etf} for sector, etf in SECTOR_ETFS.items()]))

Universe id: `sp500_50_liquid_sector_v0`; stock count: `50`.

,sector,ticker_count
0,Communication Services,5
1,Consumer Discretionary,5
2,Consumer Staples,4
3,Energy,4
4,Financials,6
5,Health Care,6
6,Industrials,5
7,Information Technology,6
8,Materials,3
9,Real Estate,3


,sector,ticker
0,Information Technology,AAPL
1,Information Technology,MSFT
2,Information Technology,NVDA
3,Information Technology,AVGO
4,Information Technology,AMD
5,Information Technology,ADBE
6,Health Care,UNH
7,Health Care,JNJ
8,Health Care,LLY
9,Health Care,MRK


,sector,etf
0,Information Technology,XLK
1,Health Care,XLV
2,Financials,XLF
3,Consumer Discretionary,XLY
4,Communication Services,XLC
5,Industrials,XLI
6,Consumer Staples,XLP
7,Energy,XLE
8,Utilities,XLU
9,Materials,XLB


## Downloader command

The command below is printed, not executed. It writes local raw and processed files under `data/`, which must stay out of git.

In [3]:
download_command = """poetry run python scripts/download_sp500_50_panel.py \\
  --start 2020-01-01 \\
  --end 2021-12-31 \\
  --output-root data \\
  --include-sector-etfs \\
  --condition-mode v3_prefix_market"""

print(f"RUN_DOWNLOAD={RUN_DOWNLOAD}")
print(download_command)
if RUN_DOWNLOAD:
    display(
        Markdown("RUN_DOWNLOAD is true, but this notebook does not execute downloader commands.")
    )

RUN_DOWNLOAD=False
poetry run python scripts/download_sp500_50_panel.py \
  --start 2020-01-01 \
  --end 2021-12-31 \
  --output-root data \
  --include-sector-etfs \
  --condition-mode v3_prefix_market


## Load a local processed panel if present

This cell is intentionally tolerant of missing local data. It should pass on clean public checkouts where no Yahoo-backed panel has been downloaded.

In [4]:
panel_available = METADATA_PATH.exists() and RAW_DATA_PATH.exists() and LABELS_PATH.exists()

if panel_available:
    metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
    data = torch.load(RAW_DATA_PATH, map_location="cpu", weights_only=True).float()
    labels = torch.load(LABELS_PATH, map_location="cpu", weights_only=True).float()
    display(Markdown(f"Loaded local processed panel from `{PROCESSED_DIR}`."))
else:
    metadata = {}
    data = None
    labels = None
    if ALLOW_MISSING_OUTPUTS:
        display(
            Markdown(
                f"No local processed panel found at `{PROCESSED_DIR}`. Data-dependent cells will be skipped."
            )
        )
    else:
        raise FileNotFoundError(f"Missing local processed panel at {PROCESSED_DIR}")

No local processed panel found at `/tmp/time-causal-vae-missing-sp50050`. Data-dependent cells will be skipped.

## Train/eval counts, labels, and prefix safety

The v3 condition mode uses start-of-window SPY/VIX features plus previous completed-window market summaries. Previous-window summaries end strictly before the generated 60-day stock-return window starts.

In [5]:
expected_v3_condition_names = [
    "spy_log_return_start",
    "log_vix_level_start",
    "previous_window_spy_realized_volatility",
    "previous_window_equal_weight_realized_volatility",
    "previous_window_average_correlation",
    "previous_window_equal_weight_log_return",
]

split_metadata = metadata.get("split", {"train_window_count": 355, "eval_window_count": 89})
condition_names = metadata.get("condition_names", expected_v3_condition_names)
lag_convention = metadata.get(
    "condition_lag_convention",
    {
        "summary": "previous_completed_window_statistics_plus_start_conditions",
        "previous_window_end_strictly_before_generated_start": True,
    },
)

display(pd.DataFrame([split_metadata]))
display(
    pd.DataFrame({
        "condition_index": range(len(condition_names)),
        "condition_name": condition_names,
    })
)
display(
    pd.DataFrame([
        {
            "condition_mode": metadata.get("condition_mode", "v3_prefix_market"),
            "no_leakage": metadata.get("no_leakage", True),
            "lag_summary": lag_convention.get("summary"),
            "previous_window_end_strictly_before_generated_start": lag_convention.get(
                "previous_window_end_strictly_before_generated_start", True
            ),
        }
    ])
)

if data is not None and labels is not None:
    display(
        pd.DataFrame([
            {
                "tensor": "raw_data",
                "shape": tuple(data.shape),
                "finite": bool(torch.isfinite(data).all()),
            },
            {
                "tensor": "labels",
                "shape": tuple(labels.shape),
                "finite": bool(torch.isfinite(labels).all()),
            },
        ])
    )

,train_window_count,eval_window_count
0,355,89


,condition_index,condition_name
0,0,spy_log_return_start
1,1,log_vix_level_start
2,2,previous_window_spy_realized_volatility
3,3,previous_window_equal_weight_realized_volatility
4,4,previous_window_average_correlation
5,5,previous_window_equal_weight_log_return


,condition_mode,no_leakage,lag_summary,previous_window_end_strictly_before_generated_start
0,v3_prefix_market,True,previous_completed_window_statistics_plus_star...,True


## Cross-sectional diagnostics

When local data is available, this section computes raw-panel covariance, correlation, eigenspectrum, and sector-block summaries. It skips cleanly without local data.

In [6]:
if data is None:
    display(Markdown("Local panel unavailable. Cross-sectional diagnostics skipped."))
else:
    returns = data.detach().cpu()
    flat_returns = returns.reshape(-1, returns.shape[-1]).numpy()
    cov = np.cov(flat_returns, rowvar=False)
    std = np.sqrt(np.clip(np.diag(cov), a_min=1e-12, a_max=None))
    corr = cov / np.outer(std, std)
    eigvals = np.linalg.eigvalsh(corr)[::-1]
    sector_ids = np.array(metadata.get("sector_label_ids", []), dtype=int)
    sector_rows = []
    if sector_ids.size == returns.shape[-1]:
        for sector_id in sorted(set(sector_ids.tolist())):
            mask = sector_ids == sector_id
            block = corr[np.ix_(mask, mask)]
            off_diag = block[~np.eye(block.shape[0], dtype=bool)]
            sector_rows.append({
                "sector_id": int(sector_id),
                "asset_count": int(mask.sum()),
                "mean_off_diagonal_corr": float(off_diag.mean()) if off_diag.size else np.nan,
            })

    display(
        pd.DataFrame([
            {"metric": "mean_asset_volatility", "value": float(std.mean())},
            {"metric": "top_corr_eigenvalue", "value": float(eigvals[0])},
            {
                "metric": "top5_corr_eigenvalue_mass",
                "value": float(eigvals[:5].sum() / eigvals.sum()),
            },
        ])
    )
    if sector_rows:
        display(pd.DataFrame(sector_rows))

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
    image = axes[0].imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
    axes[0].set_title("Panel correlation")
    fig.colorbar(image, ax=axes[0], fraction=0.046, pad=0.04)
    axes[1].plot(np.arange(1, 16), eigvals[:15], marker="o")
    axes[1].set_title("Top correlation eigenvalues")
    axes[1].set_xlabel("Rank")
    axes[1].set_ylabel("Eigenvalue")
    display(fig)
    plt.close(fig)

Local panel unavailable. Cross-sectional diagnostics skipped.

## Experimental profile metadata

The profile metadata is intentionally separate from the public model registry. This notebook reads `trained_models/multidim_profiles.yaml`, not `trained_models/model_registry.yaml`; no multidimensional public default exists.

In [7]:
profile_names = ["balanced_empirical", "discrete_reporting"]
rows = []
for profile_name in profile_names:
    selection = select_multidim_profile("sp500_50_panel", profile_name).to_dict()
    profile = selection["metadata"]
    rows.append({
        "profile": profile_name,
        "family": selection["family"],
        "candidate": profile["candidate"],
        "public_default": selection["public_default"],
        "summary": profile["evidence"]["profile_summary"],
        "caveats": ", ".join(profile.get("caveats", [])[:2]),
    })
display(pd.DataFrame(rows))

,profile,family,candidate,public_default,summary,caveats
0,balanced_empirical,continuous,beta_cvae_latent8_hidden64_v3,False,Best current one-seed empirical v3 candidate a...,"one_seed_local, Not seed-robust on empirical v3."
1,discrete_reporting,discrete,factor_pca_rvq_q2_cb64_factorised,False,Empirical discrete reporting variant only; cal...,"pair_and_regime_miscalibration, over_dispersed..."


## Non-smoke commands to run manually

These commands are printed for reference. This notebook does not run them.

In [8]:
commands = [
    download_command,
    "poetry run tcvae-train --config configs/experiments/sp500_50_panel_v3_beta_cvae_latent8_hidden64.yaml --output-dir outputs/sp500_50_panel_v3/continuous --no-wandb",
    "poetry run tcvae-train-tokenizer --config configs/experiments/sp500_50_panel_v3_factor_pca_rvq_q2_cb64_tokenizer_auxloss.yaml --output-dir outputs/sp500_50_panel_v3/tokenizer --no-wandb",
    "poetry run tcvae-train-token-prior --config configs/experiments/sp500_50_panel_v3_factor_pca_rvq_q2_cb64_prior_factorised_additive.yaml --output-dir outputs/sp500_50_panel_v3/prior --no-wandb",
]

print("Notebook guards:")
print(f"RUN_DOWNLOAD={RUN_DOWNLOAD}")
print(f"RUN_TRAINING={RUN_TRAINING}")
print(f"RUN_EVALUATION={RUN_EVALUATION}")
print()
for command in commands:
    print(command)
    print()

if RUN_DOWNLOAD or RUN_TRAINING or RUN_EVALUATION:
    display(
        Markdown("One or more guards are true, but this EDA notebook still prints commands only.")
    )

Notebook guards:
RUN_DOWNLOAD=False
RUN_TRAINING=False
RUN_EVALUATION=False

poetry run python scripts/download_sp500_50_panel.py \
  --start 2020-01-01 \
  --end 2021-12-31 \
  --output-root data \
  --include-sector-etfs \
  --condition-mode v3_prefix_market

poetry run tcvae-train --config configs/experiments/sp500_50_panel_v3_beta_cvae_latent8_hidden64.yaml --output-dir outputs/sp500_50_panel_v3/continuous --no-wandb

poetry run tcvae-train-tokenizer --config configs/experiments/sp500_50_panel_v3_factor_pca_rvq_q2_cb64_tokenizer_auxloss.yaml --output-dir outputs/sp500_50_panel_v3/tokenizer --no-wandb

poetry run tcvae-train-token-prior --config configs/experiments/sp500_50_panel_v3_factor_pca_rvq_q2_cb64_prior_factorised_additive.yaml --output-dir outputs/sp500_50_panel_v3/prior --no-wandb

